In [ ]:
import pandas as pd
import corner
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.polynomial import polyval
import emcee
import multiprocessing

In [ ]:
#Unit = Age: 3^x Myr

#Tens = Exoplanet: 1 = No, 2 = Yes

#Hundreds = Binary: 1 = No, 2 = Yes

#0 = No Data

In [ ]:
# import Ultracool Sheets with absolute magnitudes attatched
UltracoolSheets_path = "../data/empirical/UCS_with_flagged_and_AbsMag.csv"
df = pd.read_csv(UltracoolSheets_path, low_memory=False)

In [ ]:
# function pulls ofbject using flagging system
def get_objects(df, number):
    df = df.copy()
    df['total_new'] = df['total'].astype(str).str.zfill(3)
    a = '^' + number.replace('*', '.') + '$'   #essentially, ^makes sure that the number is at the start, not in the middle of a string or smth. $ makes sure that the string ends there.
    return df[df['total_new'].str.match(a)]

In [ ]:
# pulls objects older then flag given
def get_old_objects(df, number, age_min):
    df1 = pd.DataFrame()
    for age in range(age_min, 10):
        a = number.replace('*', str(age))
        df1 = pd.concat([df1, get_objects(df, a)], ignore_index=True)
    return df1

In [ ]:
# calls non-binary, non-exoplanet's over 3**6
df_old_BD = get_old_objects(df, '11*', 7)
# removes negative SpT
df_old_BD = df_old_BD[df_old_BD['sptnum_ir_formula_x'] >= 0].copy()

In [ ]:
# shows objects
plt.scatter(df_old_BD['sptnum_ir_formula_x'], df_old_BD['MH_2MASS'])

In [ ]:
def log_likelihood(params, x, y, y_err):
    # extract parameters: all but the last are polynomial coefficients, the last is the intrinsic scatter
    poly_coeffs = params[:-1]
    s = params[-1]

    # calculate the model
    y_model = polyval(x, poly_coeffs)

    # calculate log likelihood with the intrinsic scatter term
    tot_var = y_err**2 + s**2
    resid = y - y_model
    ll = -0.5 * np.sum((resid**2 / tot_var) + np.log(2 * np.pi * tot_var))
    
    return ll

In [ ]:
def log_prior(params):
    poly_coeffs = params[:-1]
    s = params[-1]

    # flat prior on coefficients to keep them within reasonable bounds
    if not np.all(np.abs(poly_coeffs) < 100.0):
        return -np.inf

    # flat prior on intrinsic scatter (must be strictly positive)
    if not 1e-6 < s < 2.0: 
        return -np.inf

    return 0.0

In [ ]:
def log_probability(params, x, y, y_err):
    # combine prior and likelihood
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf

    ll = log_likelihood(params, x, y, y_err)
    return lp + ll

In [ ]:
def calculate_model_bic(degree, x_data, y_data, y_err):
    # setup dimensions
    k = degree + 2
    n = len(x_data)

    N_DIM = k
    N_WALKERS = 4 * N_DIM
    MAX_STEPS = 5000

    # generate initial guesses using a standard weighted polyfit
    weights_polyfit = 1.0 / y_err
    try:
        initial_poly_guess = np.polyfit(x_data, y_data, degree, w=weights_polyfit)[::-1]
    except np.linalg.LinAlgError:
        print(f"Skipping Degree {degree}: np.polyfit failed (LinAlgError).")
        return np.inf, -np.inf, k, None, 0

    if np.any(~np.isfinite(initial_poly_guess)):
        print(f"Skipping Degree {degree}: Initial polyfit resulted in non-finite values.")
        return np.inf, -np.inf, k, None, 0

    # initialise walker starting positions in a tight ball around the guess
    pos = np.zeros((N_WALKERS, N_DIM))
    for i in range(N_DIM - 1):
        pos[:, i] = initial_poly_guess[i] + 1e-4 * np.random.randn(N_WALKERS)
    pos[:, -1] = 0.1 + 1e-4 * np.random.randn(N_WALKERS)

    if np.any(~np.isfinite(pos)):
        print(f"Skipping Degree {degree}: Initial MCMC positions contain non-finite values.")
        return np.inf, -np.inf, k, None, 0

    # run the mcmc sampler across available cpu cores
    with multiprocessing.Pool() as pool:
        sampler = emcee.EnsembleSampler(N_WALKERS, N_DIM, log_probability, args=(x_data, y_data, y_err), pool=pool)

        index = 0
        
        # track autocorrelation to determine convergence
        for sample in sampler.sample(pos, iterations=MAX_STEPS, thin=15):
            index += 1
            if index % 100 == 0: 
                tau = sampler.get_autocorr_time(tol=0) 
                converged = np.all(tau * 50 < sampler.iteration) 
                if converged:
                    break

    if sampler.iteration == 0:
        print(f"Skipping Degree {degree}: Sampler did not run any iterations.")
        return np.inf, -np.inf, k, None, 0

    # discard the first 20% of the chain as burn-in
    burnin = max(1, int(sampler.iteration * 0.2)) 

    if burnin >= sampler.iteration:
        print(f"Warning: Burnin ({burnin}) is too large for total iterations ({sampler.iteration}). Setting burnin to 0.")
        burnin = 0

    all_log_probs = sampler.get_log_prob(discard=burnin, flat=True)

    if all_log_probs.size == 0:
        print(f"Warning: No log probabilities after discarding burnin for Degree {degree}. Returning infinite BIC.")
        return np.inf, -np.inf, k, sampler, burnin 

    # extract the maximum likelihood to calculate the bayesian information criterion
    l_max = np.max(all_log_probs)
    bic = k * np.log(n) - 2 * l_max
    
    return bic, l_max, k, sampler, burnin 

In [ ]:
def run_polyfit_band(df, band, err):
    # mask out missing data
    mask_finite = (
        np.isfinite(df['sptnum_ir_formula_x']) &
        np.isfinite(df[band]) &
        np.isfinite(df[err])
    )

    # apply an error floor to prevent infinite weights
    MIN_ERROR_FLOOR = 1e-3

    x_data = df.loc[mask_finite, 'sptnum_ir_formula_x']
    y_data = df.loc[mask_finite, band]
    
    # safe clipping to prevent SettingWithCopyWarning
    y_err = df.loc[mask_finite, err].clip(lower=MIN_ERROR_FLOOR)

    degrees_to_test = [1, 2, 3, 4, 5, 6, 7]
    results_list = []
    n = len(x_data)

    for degree in degrees_to_test:
        print(f"\nSTARTING FIT FOR DEGREE {degree}")

        if (degree + 2) > n:
            print("Skipping: Not enough data points for this degree.")
            results_list.append({'Degree': degree, 'k': degree + 2, 'BIC': np.inf, 'L_max': -np.inf, 'Best_s': np.nan, 'Samples': np.array([])})
            continue

        # run the emcee sampler
        bic, l_max, k, sampler, burnin = calculate_model_bic(degree, x_data, y_data, y_err)

        if sampler is None or sampler.iteration == 0 or burnin >= sampler.iteration:
            print(f"WARNING: Sampler for Degree {degree} failed to run or has insufficient iterations. Samples will be empty.")
            flat_samples = np.array([])
            best_s = np.nan
        else:
            flat_samples = sampler.get_chain(discard=burnin, flat=True)

            if flat_samples.size == 0:
                print(f"WARNING: Final samples for Degree {degree} were empty after discard. Model failed to converge properly.")
                best_s = np.nan
            else:
                best_s = np.median(flat_samples[:, -1])

        results_list.append({'Degree': degree, 'k': k, 'BIC': bic, 'L_max': l_max, 'Best_s': best_s, 'Samples': flat_samples})

        if flat_samples.size > 0:
            print(f"--- MCMC Fit Results for Degree {degree} (Median & 1-sigma errors) ---")
            current_labels = [f"a_{i_param}" for i_param in range(degree + 1)]
            current_labels.append("s")

            # print the median and 1-sigma percentile bounds
            for i_param in range(len(current_labels)):
                q = np.percentile(flat_samples[:, i_param], [16, 50, 84])
                median = q[1]
                err_minus = q[1] - q[0]
                err_plus = q[2] - q[1]
                print(f"{current_labels[i_param]}: {median:.3f} (+{err_plus:.3f} / -{err_minus:.3f})")
            print("-" * 60)
        else:
            print(f"--- No valid MCMC samples for Degree {degree} to print parameters. ---")
            print("-" * 60)

    # rank the models by their bic score
    results_df = pd.DataFrame(results_list)
    results_df = results_df.sort_values(by='BIC')

    print("\n--- Final Model Comparison ---")
    print(results_df.to_string(index=False))

    best_model = results_df.iloc[0]
    print(f"\nConclusion: The optimal polynomial degree is {int(best_model['Degree'])} for the {band} band.")

    if np.isnan(best_model['Best_s']):
        print("The best-fit nuisance parameter (s) for this model is UNRELIABLE (NaN) due to sampling failure.")
    else:
        print(f"The best-fit nuisance parameter (s) for this model is {best_model['Best_s']:.4f} magnitudes.")
        
    return results_list, best_model

In [ ]:
# map the band data columns
magnitudes_2MASS = [
    ('MJ_2MASS', 'MJ_err_2MASS', 'J'), 
    ('MH_2MASS', 'MH_err_2MASS', 'H'), 
    ('MKs_2MASS', 'MKs_err_2MASS', 'Ks')
]

# magnitudes_SPITZER = [
#     ('M[3.6]', 'M[3.6]err', '[3.6]'), 
#     ('M[4.5]', 'M[4.5]err', '[4.5]')
# ]

results_list = []
best_model = []

# run the pipeline across the required bands
for band, err, sym in magnitudes_2MASS:
    results, best_mod = run_polyfit_band(df_old_BD, band, err)
    results_list.append(results)
    best_model.append(best_mod)

In [ ]:
def plot_mcmc_corner(samples, degree, band):
    # define parameter labels for the axes (a0, a1... and the scatter s)
    labels = [f"$a_{i}$" for i in range(degree + 1)] + ["$s$"]

    # generate the corner plot
    fig = corner.corner(
        samples,
        labels=labels,
        quantiles=[0.16, 0.5, 0.84],
        show_titles=True,
        title_kwargs={"fontsize": 12},
        label_kwargs={"fontsize": 14}
    )
    
    fig.suptitle(f"MCMC Posteriors for {band} Band (Degree {degree})", fontsize=16, y=1.02)

    # save directly to the repository plots folder
    plt.savefig(f'../plots/MCMC_Corner_{band}_Deg{degree}.pdf', bbox_inches='tight')
    plt.show()

def plot_best_fit_polynomial(df, band, err_col, samples, degree):
    plt.figure(figsize=(10, 6))

    # filter for valid data points to plot the scatter
    mask = np.isfinite(df['sptnum_ir_formula_x']) & np.isfinite(df[band]) & np.isfinite(df[err_col])
    x_data = df.loc[mask, 'sptnum_ir_formula_x']
    y_data = df.loc[mask, band]
    y_err = df.loc[mask, err_col]

    # plot the raw empirical brown dwarf data
    plt.errorbar(x_data, y_data, yerr=y_err, fmt='o', color='black', alpha=0.4, markersize=4, label='Empirical Data')

    # generate a smooth x-axis for the polynomial lines
    x_fit = np.linspace(x_data.min(), x_data.max(), 200)

    # pick 100 random samples from the mcmc chain to show the uncertainty spread
    random_indices = np.random.randint(len(samples), size=100)
    for idx in random_indices:
        # extract all coefficients (ignoring the final 's' parameter)
        poly_coeffs = samples[idx, :-1]
        plt.plot(x_fit, polyval(x_fit, poly_coeffs), color='orange', alpha=0.05, zorder=1)

    # calculate and plot the absolute median best-fit line on top
    median_coeffs = np.median(samples[:, :-1], axis=0)
    plt.plot(x_fit, polyval(x_fit, median_coeffs), color='red', linewidth=2.5, label=f'Median Fit (Deg {degree})', zorder=2)

    # formatting
    plt.xlabel("Spectral Type (SpT Number)", fontsize=14)
    plt.ylabel(f"Absolute Magnitude ($M_{{{band}}}$)", fontsize=14)
    plt.title(f"Empirical Brown Dwarf Sequence: {band} Band", fontsize=16)
    
    # astronomically standard to invert magnitude axes
    plt.gca().invert_yaxis()
    plt.legend(loc='upper left', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.5)

    # save directly to the repository plots folder
    plt.savefig(f'../plots/Empirical_Fit_{band}_Deg{degree}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# loop through the best models we saved during the MCMC run
for best_mod, (band, err, sym) in zip(best_model, magnitudes_2MASS):
    deg = int(best_mod['Degree'])
    samples = best_mod['Samples']

    # only plot if the sampler successfully converged and returned data
    if samples.size > 0:
        print(f"Generating plots for {sym} band...")
        plot_mcmc_corner(samples, deg, sym)
        plot_best_fit_polynomial(df_old_BD, band, err, samples, deg)